In [0]:
%pip install torch torchvision torchaudio


In [0]:
import os
import torch
from PIL import Image
from torch.utils.data import Dataset

class FishDataset(Dataset):
    def __init__(self, img_dir, annotations, transforms=None):
        self.img_dir = img_dir
        self.annotations = annotations  # list of dicts with boxes and labels
        self.transforms = transforms

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.annotations[idx]['filename'])
        img = Image.open(img_path).convert("RGB")
        boxes = torch.tensor(self.annotations[idx]['boxes'], dtype=torch.float32)
        labels = torch.tensor(self.annotations[idx]['labels'], dtype=torch.int64)
        target = {"boxes": boxes, "labels": labels}
        if self.transforms:
            img = self.transforms(img)
        return img, target

    def __len__(self):
        return len(self.annotations)


In [0]:
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

model = torchvision.models.detection.fasterrcnn_resnet50_fpn(pretrained=True)

# Replace the classifier with the number of classes (1 fish + background)
num_classes = 2
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
